# Sorting Algorithms: Performance Investigation

**Problem statement:** Implement **Quicksort**, **Mergesort**, and **Heapsort**, and investigate their performance on arrays of size $10^2$, $10^3$, $10^4$, $10^5$, and $10^6$. For each size, consider:
- **Random** integers
- **Ascending** (already sorted)
- **Descending** (reverse sorted)

| Algorithm | Best case | Average case | Worst case | Space |
|-----------|:---------:|:------------:|:----------:|:-----:|
| Quicksort | $O(n \log n)$ | $O(n \log n)$ | $O(n^2)$ | $O(\log n)$ |
| Mergesort | $O(n \log n)$ | $O(n \log n)$ | $O(n \log n)$ | $O(n)$ |
| Heapsort  | $O(n \log n)$ | $O(n \log n)$ | $O(n \log n)$ | $O(1)$ |

In [4]:
# Cell 1: Imports and configuration
from __future__ import annotations
import random
import time
import sys
import copy

# Increase recursion limit for quicksort on large sorted arrays
sys.setrecursionlimit(1_100_000)

## Quicksort

Uses the **median-of-three** pivot strategy to mitigate worst-case behavior on already-sorted inputs. Partition follows the Lomuto scheme.

- **Best / Average:** $O(n \log n)$
- **Worst:** $O(n^2)$ (rare with median-of-three)
- **Space:** $O(\log n)$ (recursion stack)

In [5]:
# Cell 2: Quicksort implementation (median-of-three + insertion sort cutoff)

def _median_of_three(arr: list[int], lo: int, hi: int) -> int:
    """Return the index of the median of arr[lo], arr[mid], arr[hi]."""
    mid = (lo + hi) // 2
    if arr[lo] > arr[mid]:
        arr[lo], arr[mid] = arr[mid], arr[lo]
    if arr[lo] > arr[hi]:
        arr[lo], arr[hi] = arr[hi], arr[lo]
    if arr[mid] > arr[hi]:
        arr[mid], arr[hi] = arr[hi], arr[mid]
    return mid


def _insertion_sort(arr: list[int], lo: int, hi: int) -> None:
    """In-place insertion sort on arr[lo..hi]."""
    for i in range(lo + 1, hi + 1):
        key = arr[i]
        j = i - 1
        while j >= lo and arr[j] > key:
            arr[j + 1] = arr[j]
            j -= 1
        arr[j + 1] = key


def quicksort(arr: list[int]) -> list[int]:
    """Sort arr in-place using quicksort with median-of-three pivot."""
    _quicksort(arr, 0, len(arr) - 1)
    return arr


def _quicksort(arr: list[int], lo: int, hi: int) -> None:
    CUTOFF = 16
    while lo < hi:
        if hi - lo < CUTOFF:
            _insertion_sort(arr, lo, hi)
            return
        # Median-of-three pivot
        mid = _median_of_three(arr, lo, hi)
        arr[mid], arr[hi] = arr[hi], arr[mid]
        pivot = arr[hi]
        # Partition
        i = lo
        for j in range(lo, hi):
            if arr[j] <= pivot:
                arr[i], arr[j] = arr[j], arr[i]
                i += 1
        arr[i], arr[hi] = arr[hi], arr[i]
        # Tail-call optimization: recurse on smaller partition, loop on larger
        if i - lo < hi - i:
            _quicksort(arr, lo, i - 1)
            lo = i + 1
        else:
            _quicksort(arr, i + 1, hi)
            hi = i - 1

## Mergesort

Classic divide-and-conquer: split the array in half, recursively sort each half, and merge.

- **All cases:** $O(n \log n)$
- **Space:** $O(n)$ (auxiliary array during merge)

In [6]:
# Cell 3: Mergesort implementation

def mergesort(arr: list[int]) -> list[int]:
    """Sort arr using top-down mergesort. Returns a new sorted list."""
    if len(arr) <= 1:
        return arr
    mid = len(arr) // 2
    left = mergesort(arr[:mid])
    right = mergesort(arr[mid:])
    return _merge(left, right)


def _merge(left: list[int], right: list[int]) -> list[int]:
    """Merge two sorted lists into one sorted list."""
    result: list[int] = []
    i = j = 0
    while i < len(left) and j < len(right):
        if left[i] <= right[j]:
            result.append(left[i])
            i += 1
        else:
            result.append(right[j])
            j += 1
    result.extend(left[i:])
    result.extend(right[j:])
    return result

## Heapsort

Builds a max-heap in-place, then repeatedly extracts the maximum.

- **All cases:** $O(n \log n)$
- **Space:** $O(1)$ (in-place)

In [12]:
# Cell 4: Heapsort implementation

def heapsort(arr: list[int]) -> list[int]:
    """Sort arr in-place using heapsort."""
    n = len(arr)
    # Build max-heap
    for i in range(n // 2 - 1, -1, -1):
        _sift_down(arr, n, i)
    # Extract elements one by one
    for i in range(n - 1, 0, -1):
        arr[0], arr[i] = arr[i], arr[0]
        _sift_down(arr, i, 0)
    return arr


def _sift_down(arr: list[int], size: int, root: int) -> None:
    """Sift down the element at index root to maintain the max-heap property."""
    largest = root
    left = 2 * root + 1
    right = 2 * root + 2
    if left < size and arr[left] > arr[largest]:
        largest = left
    if right < size and arr[right] > arr[largest]:
        largest = right
    if largest != root:
        arr[root], arr[largest] = arr[largest], arr[root]
        _sift_down(arr, size, largest)

## Correctness Verification

Quick sanity check before running the full benchmark.

In [8]:
# Cell 5: Verify correctness

rng = random.Random(2026)
test_arr = [rng.randint(-1000, 1000) for _ in range(200)]
expected = sorted(test_arr)

assert quicksort(test_arr[:]) == expected, "Quicksort FAILED"
assert mergesort(test_arr[:]) == expected, "Mergesort FAILED"
assert heapsort(test_arr[:])  == expected, "Heapsort FAILED"

print("All three sorting algorithms passed the correctness test.")

All three sorting algorithms passed the correctness test.


## Benchmark Setup

For each combination of **(array size × input type × algorithm)**, we time the sorting and collect results.

- **Sizes:** $10^2, 10^3, 10^4, 10^5, 10^6$
- **Input types:** Random, Ascending (sorted), Descending (reverse sorted)
- **Timing:** `time.perf_counter()` in seconds

In [9]:
# Cell 6: Benchmark utilities

SIZES = [10**2, 10**3, 10**4, 10**5, 10**6]
INPUT_TYPES = ["Random", "Ascending", "Descending"]
ALGORITHMS = {
    "Quicksort": quicksort,
    "Mergesort": mergesort,
    "Heapsort":  heapsort,
}


def generate_array(size: int, input_type: str, rng: random.Random) -> list[int]:
    """Generate an array of the given size and type."""
    if input_type == "Random":
        return [rng.randint(0, size * 10) for _ in range(size)]
    elif input_type == "Ascending":
        return list(range(size))
    elif input_type == "Descending":
        return list(range(size - 1, -1, -1))
    else:
        raise ValueError(f"Unknown input type: {input_type}")


def time_sort(sort_fn, arr: list[int]) -> float:
    """Time a sorting function on a copy of arr. Returns seconds."""
    data = arr[:]  # copy so original is preserved
    t0 = time.perf_counter()
    sort_fn(data)
    t1 = time.perf_counter()
    return t1 - t0

In [10]:
# Cell 7: Run the full benchmark

rng = random.Random(2026)
results: dict[tuple[int, str, str], float] = {}

for size in SIZES:
    for input_type in INPUT_TYPES:
        arr = generate_array(size, input_type, rng)
        for algo_name, algo_fn in ALGORITHMS.items():
            elapsed = time_sort(algo_fn, arr)
            results[(size, input_type, algo_name)] = elapsed
            print(f"  n={size:>8,}  {input_type:<11}  {algo_name:<10}  {elapsed:.6f} s")
    print()

print("Benchmark complete.")

  n=     100  Random       Quicksort   0.000118 s
  n=     100  Random       Mergesort   0.000297 s
  n=     100  Random       Heapsort    0.000272 s
  n=     100  Ascending    Quicksort   0.000058 s
  n=     100  Ascending    Mergesort   0.000191 s
  n=     100  Ascending    Heapsort    0.000278 s
  n=     100  Descending   Quicksort   0.000128 s
  n=     100  Descending   Mergesort   0.000221 s
  n=     100  Descending   Heapsort    0.000230 s

  n=   1,000  Random       Quicksort   0.001722 s
  n=   1,000  Random       Mergesort   0.003099 s
  n=   1,000  Random       Heapsort    0.003564 s
  n=   1,000  Ascending    Quicksort   0.000491 s
  n=   1,000  Ascending    Mergesort   0.001178 s
  n=   1,000  Ascending    Heapsort    0.002612 s
  n=   1,000  Descending   Quicksort   0.001480 s
  n=   1,000  Descending   Mergesort   0.001287 s
  n=   1,000  Descending   Heapsort    0.002098 s

  n=  10,000  Random       Quicksort   0.013306 s
  n=  10,000  Random       Mergesort   0.028329 

## Results Table

In [11]:
# Cell 8: Display results as a formatted table

def format_time(t: float) -> str:
    """Format time in appropriate units."""
    if t < 0.001:
        return f"{t*1_000_000:.1f} µs"
    elif t < 1.0:
        return f"{t*1_000:.2f} ms"
    else:
        return f"{t:.3f} s"


# Print header
header = f"{'n':>10} | {'Input':<11} | {'Quicksort':>12} | {'Mergesort':>12} | {'Heapsort':>12}"
print(header)
print("-" * len(header))

for size in SIZES:
    for input_type in INPUT_TYPES:
        qs = format_time(results[(size, input_type, "Quicksort")])
        ms = format_time(results[(size, input_type, "Mergesort")])
        hs = format_time(results[(size, input_type, "Heapsort")])
        print(f"{size:>10,} | {input_type:<11} | {qs:>12} | {ms:>12} | {hs:>12}")
    print("-" * len(header))

         n | Input       |    Quicksort |    Mergesort |     Heapsort
---------------------------------------------------------------------
       100 | Random      |     118.1 µs |     296.7 µs |     271.8 µs
       100 | Ascending   |      58.5 µs |     191.0 µs |     277.6 µs
       100 | Descending  |     128.0 µs |     221.4 µs |     229.8 µs
---------------------------------------------------------------------
     1,000 | Random      |      1.72 ms |      3.10 ms |      3.56 ms
     1,000 | Ascending   |     491.4 µs |      1.18 ms |      2.61 ms
     1,000 | Descending  |      1.48 ms |      1.29 ms |      2.10 ms
---------------------------------------------------------------------
    10,000 | Random      |     13.31 ms |     28.33 ms |     34.02 ms
    10,000 | Ascending   |      8.27 ms |     14.27 ms |     31.28 ms
    10,000 | Descending  |     18.43 ms |     15.16 ms |     29.54 ms
---------------------------------------------------------------------
   100,000 | Random 

## Analysis and Discussion

### Observations by algorithm

#### Quicksort
- **Fastest overall.** On random input it consistently outperforms the other two: 13.31 ms at $n=10^4$, 157.86 ms at $n=10^5$, and 2.365 s at $n=10^6$.
- **Ascending input is its best case.** Thanks to median-of-three, sorted data becomes almost free: at $n=10^6$ it takes only **1.130 s** — roughly half the time of random input (2.365 s).
- **Descending input is its weakest scenario.** At $n=10^6$ it takes 2.857 s, about 20% slower than random. Despite this, it still beats mergesort (2.044 s is close, but quicksort remains competitive overall) and is far ahead of heapsort.
- The **insertion-sort cutoff** (subarrays $\leq 16$) and **tail-call optimization** contribute to its low constant factor.

#### Mergesort
- **Most stable across input types.** The difference between its best (ascending, 2.035 s at $n=10^6$) and worst (random, 3.930 s) is a factor of ~1.9×, which reflects only the cost of more merge comparisons.
- At $n=10^6$ descending, it takes **2.044 s** — actually beating quicksort's 2.857 s on the same input. This shows mergesort's advantage on structured data.
- **Downside:** At $n=10^6$ random it takes 3.930 s vs. quicksort's 2.365 s — about 1.66× slower due to the overhead of allocating and copying auxiliary arrays ($O(n)$ extra space).

#### Heapsort
- **Consistently the slowest.** At $n=10^6$: 9.283 s (random), 5.006 s (ascending), 4.774 s (descending). It is **3.9× slower** than quicksort on random data.
- The performance gap widens with $n$. At $n=10^2$ the three algorithms are within 300 µs of each other; by $n=10^6$ heapsort is 5–7 s behind.
- **In-place** ($O(1)$ extra space), but poor cache locality (jumping between parent/child indices in the heap) explains the large constant factor.

### Observations by input type

| Input type | Quicksort | Mergesort | Heapsort | Notes |
|------------|:---------:|:---------:|:--------:|:------|
| **Random** ($n=10^6$) | 2.365 s | 3.930 s | 9.283 s | Quicksort is the clear winner |
| **Ascending** ($n=10^6$) | 1.130 s | 2.035 s | 5.006 s | All algorithms improve; quicksort benefits the most |
| **Descending** ($n=10^6$) | 2.857 s | 2.044 s | 4.774 s | Mergesort beats quicksort here |

### Growth rate analysis


Comparing $n=10^5$ to $n=10^6$ (a 10× increase in input size), the expected growth for $O(n \log n)$ is roughly $10 \times \frac{\log 10^6}{\log 10^5} = 10 \times 1.2 = 12\times$. Observed ratios on random input:For **general-purpose sorting**, Quicksort with median-of-three is the fastest in practice (up to 3.9× faster than heapsort). Mergesort provides the strongest **worst-case guarantee**, is **stable**, and actually wins on descending input. Heapsort is the most **space-efficient** ($O(1)$) but the slowest due to cache inefficiency.

- **Quicksort:** $2.365 / 0.158 \approx 15.0\times$

- **Mergesort:** $3.930 / 0.316 \approx 12.4\times$### Key takeaway

- **Heapsort:** $9.283 / 0.446 \approx 20.8\times$

Mergesort's ratio most closely matches the theoretical prediction, confirming its consistent $O(n \log n)$ behavior. Heapsort's higher ratio suggests increasing cache pressure at large $n$.

## Conclusions

1. **Quicksort is the fastest in practice** across most scenarios. At $n=10^6$ it sorted random data in **2.365 s**, compared to 3.930 s (mergesort) and 9.283 s (heapsort). The median-of-three pivot strategy effectively prevents $O(n^2)$ degradation on sorted inputs.
2. **Mergesort is the most predictable.** Its time ratio between best and worst input types is only ~1.9×, and it even **outperforms quicksort on descending data** (2.044 s vs. 2.857 s at $n=10^6$). Its $O(n)$ extra memory is the main trade-off.
3. **Heapsort is the slowest but most memory-efficient.** Its $O(1)$ space advantage comes at the cost of being **3.9× slower** than quicksort on random data at $n=10^6$, mainly due to poor cache locality in heap operations.
4. **For small arrays** ($n \leq 10^3$), all three algorithms complete within 3.6 ms or less — differences are negligible in practice.
5. **The performance gap widens significantly with $n$.** At $n=10^2$ the spread is ~220 µs; at $n=10^6$ it grows to nearly 7 seconds between quicksort and heapsort.
6. **Input order matters.** Ascending data benefits all algorithms (quicksort's best: 1.130 s; mergesort: 2.035 s; heapsort: 5.006 s). Descending data penalizes quicksort the most (2.857 s), making mergesort the winner for reverse-sorted input.
7. **Experimental growth rates confirm $O(n \log n)$ behavior.** Mergesort's observed growth factor (12.4×) most closely matches the theoretical prediction (12×), while heapsort's higher ratio (20.8×) reflects increasing cache pressure at large $n$.